In [1]:
import pandas as pd
import getpass
from sqlalchemy import create_engine, text, URL

password = getpass.getpass("Please enter your SQL password: ")

url = URL.create(
    "mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    port=3306,
    database="bookworms"
)

engine = create_engine(url)

# Test the connection.
with engine.connect() as connection:
    rows = connection.execute(text("SHOW TABLES")).fetchall()

print(rows)

ModuleNotFoundError: No module named 'sqlalchemy'

In [ ]:

df = pd.read_csv(r'C:\Users\Preeti\Desktop\Ironhack\week_4\Day_1\first_project\data\clean\bookworms_cleaned_data_file.csv')

In [ ]:
df.head()


In [ ]:
df.info()

In [ ]:
cols = [
    "title", "average_rating", "language_code", "num_pages",
    "ratings_count", "text_reviews_count", "publication_year"
]

data = df.rename(columns=str.strip)[cols].copy()

data["num_pages"] = data["num_pages"].astype("Int64")
data["publication_year"] = data["publication_year"].astype("Int64")

data = data.astype(object).where(pd.notna(data), None)

with engine.begin() as connection:
    data.to_sql(
        "books",
        con=connection,
        if_exists="append",
        index=False,
        chunksize=1000
    )

In [ ]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT * FROM books LIMIT 5")
    )
    rows = result.fetchall()

rows

In [ ]:
from sqlalchemy import text

# Prepare the same book columns used in your previous upload
cols = [
    "title", "average_rating", "language_code", "num_pages",
    "ratings_count", "text_reviews_count", "publication_year"
]

source = df.rename(columns=str.strip)[cols + ["authors"]].copy()

with engine.begin() as connection:

    # 1. Find each book's actual database ID
    stored_books = pd.read_sql("SELECT * FROM books", connection)

    matched = source.merge(
        stored_books[["book_id"] + cols],
        on=cols,
        how="left",
        validate="one_to_one"
    )

    if matched["book_id"].isna().any():
        raise ValueError("Some CSV books could not be matched to the database.")

    # 2. Separate authors: assumes multiple names are separated by "/"
    pairs = matched[["book_id", "authors"]].dropna(subset=["authors"]).copy()
    pairs["author_name"] = pairs["authors"].str.split("/")
    pairs = pairs.explode("author_name")
    pairs["author_name"] = pairs["author_name"].str.strip()
    pairs = pairs[pairs["author_name"] != ""]

    # Populate authors; MySQL generates author_id
    author_data = pairs[["author_name"]].drop_duplicates()
    author_data.to_sql(
        "authors", connection, if_exists="append", index=False
    )

    # 3. Retrieve author IDs and populate book_authors
    stored_authors = pd.read_sql("SELECT * FROM authors", connection)

    links = pairs.merge(
        stored_authors, on="author_name", validate="many_to_one"
    )

    links[["book_id", "author_id"]].drop_duplicates().to_sql(
        "book_authors", connection, if_exists="append", index=False
    )

    # 4. Define example collections and their membership rules
    rules = [
        (
            "Popular Favourites",
            "Rating >= 4.0; at least 10,000 ratings",
            "Offer books with strong reader interest",
            (matched["average_rating"] >= 4.0)
            & (matched["ratings_count"] >= 10000)
        ),
        (
            "Highly Rated Short Reads",
            "Rating >= 4.0; at least 100 ratings; 1–299 pages",
            "Offer shorter books for readers with limited time",
            (matched["average_rating"] >= 4.0)
            & (matched["ratings_count"] >= 100)
            & matched["num_pages"].between(1, 299)
        ),
        (
            "Hidden Gem Candidates",
            "Rating >= 4.0; between 100 and 9,999 ratings",
            "Help readers discover less widely rated books",
            (matched["average_rating"] >= 4.0)
            & matched["ratings_count"].between(100, 9999)
        )
    ]

    # Populate collections and collection_of_books
    for name, criteria, purpose, mask in rules:
        result = connection.execute(
            text("""
                INSERT INTO collections
                    (collection_name, selection_criteria, business_purpose)
                VALUES (:name, :criteria, :purpose)
            """),
            {"name": name, "criteria": criteria, "purpose": purpose}
        )

        members = matched.loc[mask, ["book_id"]].drop_duplicates().copy()
        members["collection_id"] = result.lastrowid

        members[["collection_id", "book_id"]].to_sql(
            "collection_of_books",
            connection,
            if_exists="append",
            index=False
        )

print("Remaining tables populated successfully.")